# 00 — Session Setup

**Run this notebook at the start of EVERY Colab session.**

Takes 5–8 minutes. When it finishes, you can run the pipeline or demo.

---

### ✅ Checklist before running
Make sure you have these Colab Secrets set (click the 🔑 key icon in the left sidebar):

| Secret name | What it is |
|-------------|------------|
| `ANTHROPIC_API_KEY` | Your Claude API key from console.anthropic.com |
| `HF_TOKEN` | HuggingFace token from huggingface.co/settings/tokens |
| `GITHUB_PAT` | GitHub Personal Access Token (repo scope, 90-day expiry) |
| `OPENAI_API_KEY` | OpenAI key (optional — only if using GPT as teacher) |

See `COLAB_SETUP_README.md` in the repo for how to get each key.

In [ ]:
# ── CELL 1: Verify GPU ────────────────────────────────────────────────────────
# Expected: Tesla T4, 15360 MiB
# If you see K80 or no GPU: Runtime → Disconnect → Reconnect
!nvidia-smi
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── CELL 2: Mount Google Drive ────────────────────────────────────────────────
# A browser popup will ask for permission — click Allow.
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted at /content/drive')

In [ ]:
# ── CELL 3: Create Drive folder structure (safe to run multiple times) ────────
import os

DRIVE_ROOT = '/content/drive/MyDrive/slm-distillation'

dirs = [
    f'{DRIVE_ROOT}/data/raw',
    f'{DRIVE_ROOT}/data/processed',
    f'{DRIVE_ROOT}/data/checkpoints',
    f'{DRIVE_ROOT}/outputs',
    f'{DRIVE_ROOT}/hf_cache',
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

print('✅ Drive folders ready:')
for d in dirs:
    print(f'   {d}')

In [ ]:
# ── CELL 4: Point HuggingFace cache to Drive ──────────────────────────────────
# Models download once and are reused across sessions — no re-downloading.
import os
os.environ['HF_HOME'] = f'{DRIVE_ROOT}/hf_cache'
print(f'✅ HF model cache → {os.environ["HF_HOME"]}')

In [ ]:
# ── CELL 5: Load API keys from Colab Secrets ──────────────────────────────────
# Keys are read from the 🔑 Secrets panel — never hardcoded here.
import os
from google.colab import userdata

def load_secret(name, required=True):
    try:
        val = userdata.get(name)
        if val:
            os.environ[name] = val
            print(f'  ✅ {name} loaded')
        else:
            print(f'  ⚠️  {name} is empty')
    except Exception:
        if required:
            print(f'  ❌ {name} not found — add it in the 🔑 Secrets panel')
        else:
            print(f'  ℹ️  {name} not set (optional)')

print('Loading secrets:')
load_secret('ANTHROPIC_API_KEY', required=True)
load_secret('HF_TOKEN',          required=True)
load_secret('OPENAI_API_KEY',    required=False)
print('\nIf any required secret shows ❌, stop and add it before continuing.')

In [ ]:
# ── CELL 6: Install dependencies ──────────────────────────────────────────────
# ~3-4 minutes. Run once per session.
print('Installing base dependencies...')
!pip install -q -r /content/project/requirements.txt 2>&1 | tail -3
print('Installing Colab GPU dependencies (Unsloth, bitsandbytes)...')
!pip install -q -r /content/project/requirements_colab.txt 2>&1 | tail -3
print('✅ All dependencies installed')

---
**⚠️ Run Cell 6 AFTER Cell 7 on first setup** (you need the repo cloned first to read requirements files).

On subsequent sessions, run Cell 7 first (pull latest code), then Cell 6 (install deps).

---

In [ ]:
# ── CELL 7: Clone or pull the GitHub repo ────────────────────────────────────
# *** CHANGE BRANCH_NAME to your pair's branch before running ***
import os, subprocess
from google.colab import userdata

BRANCH_NAME = 'main'   # ← CHANGE THIS to your branch, e.g. 'pair-1/smollm2-1.7b'
REPO_ORG    = 'Break-Through-Tech'
REPO_NAME   = 'Automation-Anywhere-1A-domain-specific-theme-labeling-via-slm-distillation'
PROJECT_DIR = '/content/project'

PAT      = userdata.get('GITHUB_PAT')
REPO_URL = f'https://{PAT}@github.com/{REPO_ORG}/{REPO_NAME}.git'

if not os.path.exists(PROJECT_DIR):
    print(f'Cloning repo (branch: {BRANCH_NAME}) ...')
    subprocess.run(['git', 'clone', '-b', BRANCH_NAME, REPO_URL, PROJECT_DIR],
                   capture_output=True, check=True)
    print('✅ Repository cloned')
else:
    print('Repository already cloned — pulling latest changes ...')
    subprocess.run(['git', '-C', PROJECT_DIR, 'checkout', BRANCH_NAME],
                   capture_output=True, check=True)
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull', 'origin', BRANCH_NAME],
                   capture_output=True, check=True)
    print(f'✅ Branch {BRANCH_NAME} up to date')

# Configure git identity (needed for pushing)
subprocess.run(['git', '-C', PROJECT_DIR, 'config', 'user.email', 'student@cornell.edu'],
               capture_output=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'config', 'user.name', 'Student Name'],
               capture_output=True)

os.chdir(PROJECT_DIR)
import sys
sys.path.insert(0, PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ── CELL 8: Verify setup ─────────────────────────────────────────────────────
import os

checks = {
    'GPU available':         'torch.cuda.is_available()',
    'ANTHROPIC_API_KEY set': '"ANTHROPIC_API_KEY" in os.environ',
    'HF_TOKEN set':          '"HF_TOKEN" in os.environ',
    'Drive mounted':         'os.path.exists("/content/drive/MyDrive")',
    'Repo cloned':           'os.path.exists("/content/project/main.py")',
    'HF cache on Drive':     'os.environ.get("HF_HOME", "").startswith("/content/drive")',
}

import torch
all_ok = True
print('Setup verification:')
for label, expr in checks.items():
    ok = eval(expr)
    icon = '✅' if ok else '❌'
    print(f'  {icon} {label}')
    if not ok:
        all_ok = False

print()
if all_ok:
    print('🚀 All checks passed! Ready to run the pipeline.')
else:
    print('⚠️  Some checks failed — fix them before running the pipeline.')

---
## Run the pipeline

After the setup cells pass, use one of the cells below.

In [ ]:
# ── Full training + evaluation pipeline ──────────────────────────────────────
# Runs all steps: cluster → label → fine-tune → evaluate
# Expected time: 30-90 minutes depending on model size
!python main.py \
    --phase 1 \
    --config configs/phase1_config.yaml \
    --device_mode colab

In [ ]:
# ── Skip training, re-run evaluation only ────────────────────────────────────
# Use this when the model is already trained and you want to re-evaluate.
# Replace the path with your actual run directory.
EXISTING_RUN = 'outputs/20260820_0014_SmolLM2-360M-Instruct_ep2'  # ← UPDATE THIS

!python main.py \
    --phase 1 \
    --config configs/phase1_config.yaml \
    --device_mode colab

In [ ]:
# ── Live demo mode ────────────────────────────────────────────────────────────
# Interactive demo: models load once, then you provide ticket files one by one.
# Replace the adapter path with your actual run directory.
ADAPTER_DIR = 'outputs/20260820_0014_SmolLM2-360M-Instruct_ep2/models/lora_adapter'  # ← UPDATE

!python main.py \
    --phase 1 \
    --config configs/phase1_config.yaml \
    --device_mode colab \
    --mode demo \
    --adapter_dir "$ADAPTER_DIR"

In [ ]:
# ── Create master CSV (optional) ──────────────────────────────────────────────
# Joins all four pivot CSVs into one wide file for analysis.
RUN_DIR = 'outputs/20260820_0014_SmolLM2-360M-Instruct_ep2'  # ← UPDATE THIS

!python main.py \
    --phase 1 \
    --config configs/phase1_config.yaml \
    --create_master_csv \
    --run_dir "$RUN_DIR"

In [ ]:
# ── Save your code changes to GitHub ─────────────────────────────────────────
import subprocess

COMMIT_MSG = 'Add experiment results for SmolLM2-360M ep2'  # ← describe your changes

result = subprocess.run(
    ['git', 'add', '.'],
    capture_output=True, text=True, cwd='/content/project'
)
result = subprocess.run(
    ['git', 'commit', '-m', COMMIT_MSG],
    capture_output=True, text=True, cwd='/content/project'
)
print(result.stdout or result.stderr)
result = subprocess.run(
    ['git', 'push'],
    capture_output=True, text=True, cwd='/content/project'
)
print(result.stdout or result.stderr or '✅ Pushed to GitHub')

---
## First session only — create your branch

Run the cell below **once** when you first set up your Colab.
Skip it in all future sessions.

In [ ]:
# ── FIRST SESSION ONLY: create your branch ───────────────────────────────────
# Change BRANCH_NAME, then run this cell once.
import subprocess

BRANCH_NAME = 'pair-X/model-name'   # ← CHANGE THIS (e.g. 'pair-1/smollm2-1.7b')

subprocess.run(['git', '-C', '/content/project', 'checkout', '-b', BRANCH_NAME], check=True)
subprocess.run(['git', '-C', '/content/project', 'push', '-u', 'origin', BRANCH_NAME], check=True)
print(f"✅ Branch '{BRANCH_NAME}' created and pushed.")
print("Now update BRANCH_NAME in Cell 7 and run normally next session.")